In [1]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ==========================================================
# DEVICE CHECK
# ==========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device :", device)

Using Device : cuda


In [3]:
# ==========================================================
# DATASET PATH
# ==========================================================

PROJECT_ROOT = Path.cwd().parent

TRAIN_DIR = PROJECT_ROOT / "data" / "augmented_train"

print("Training Dataset :", TRAIN_DIR)

Training Dataset : d:\Railway_AI_Inspector\data\augmented_train


In [4]:
# ==========================================================
# IMAGE TRANSFORMATIONS
# ==========================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

print("Transforms Created Successfully")

Transforms Created Successfully


In [5]:
# ==========================================================
# LOAD DATASET
# ==========================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)

print("Dataset Loaded Successfully")
print("Total Images :", len(train_dataset))
print("Classes :", train_dataset.classes)

Dataset Loaded Successfully
Total Images : 570
Classes : ['broken_rail', 'crack', 'misalignment', 'normal', 'surface_wear']


In [6]:
# ==========================================================
# CREATE DATALOADER
# ==========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

print("DataLoader Created Successfully")
print("Total Batches :", len(train_loader))

DataLoader Created Successfully
Total Batches : 36


In [7]:
# ==========================================================
# LOAD PRETRAINED RESNET18
# ==========================================================

model = models.resnet18(weights="DEFAULT")

num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(train_dataset.classes)
)

model = model.to(device)

print("ResNet18 Loaded Successfully")

ResNet18 Loaded Successfully


In [8]:
# ==========================================================
# LOSS FUNCTION & OPTIMIZER
# ==========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss Function and Optimizer Ready")

Loss Function and Optimizer Ready


In [9]:
# ==========================================================
# TRAIN RESNET18 ON AUGMENTED DATASET
# ==========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

print("\nTraining Complete")

Epoch [1/5] Loss: 1.0264 Accuracy: 64.04%
Epoch [2/5] Loss: 0.4563 Accuracy: 84.56%
Epoch [3/5] Loss: 0.2626 Accuracy: 91.58%
Epoch [4/5] Loss: 0.2785 Accuracy: 90.53%
Epoch [5/5] Loss: 0.2289 Accuracy: 92.11%

Training Complete
